<a href="https://colab.research.google.com/github/usami0jp/ai2023-old/blob/main/SFT_initial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# LLM Fine-Tuning (QLoRA) 一括実行プログラム
# 対象環境: Google Colab (ランタイム: T4 GPU 推奨)

# 1. 必要なライブラリのインストール
print("--- Installing libraries ---")
!pip install -q transformers datasets accelerate peft trl bitsandbytes

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

# 2. 基本設定
print("--- Config 設定 ---")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 3. トークナイザーと量子化(4bit)の設定
print("--- Loading Model and Tokenizer ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# T4 GPU用に float16 に設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 4. 教師データの準備 (チャット形式をテキスト化)
print("--- Preparing Dataset ---")
data = [
    {"messages": [{"role": "user", "content": "こんにちは!"}, {"role": "assistant", "content": "ふん、挨拶くらいはしてあげようかしら。こんにちは。"}]},
    {"messages": [{"role": "user", "content": "お腹が空きました。"}, {"role": "assistant", "content": "別にアンタのために用意したんじゃないんだからね! これ食べなさいよ!"}]},
    {"messages": [{"role": "user", "content": "おすすめのプログラミング言語は?"}, {"role": "assistant", "content": "そんなのPython に決まってるじゃない! 初心者でも扱いやすいんだから!"}]},
]

formatted_data = []
for item in data:
    formatted_text = tokenizer.apply_chat_template(item["messages"], tokenize=False)
    formatted_data.append({"text": formatted_text})

dataset = Dataset.from_list(formatted_data)

# モデル読み込み
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# キャッシュ無効化・勾配チェックポイント有効化
model.config.use_cache = False
model.gradient_checkpointing_enable()

# 5. LoRA (PEFT) の設定
print("--- Configuring LoRA ---")
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 6. トレーナーの設定とファインチューニングの実行
print("--- Starting Training ---")
training_args = TrainingArguments(
    output_dir="./results",
    max_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    disable_tqdm=True,
    fp16=True,         # T4 GPU用
    bf16=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=512
)

# 学習実行
trainer.train()
print("--- Training Completed ---")

# 7. 学習後の推論テスト
print("--- Testing Model ---")
instruction = [{"role": "user", "content": "こんにちは、自己紹介してください。"}]
prompt = tokenizer.apply_chat_template(instruction, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 推論時はキャッシュを有効化
model.config.use_cache = True
outputs = model.generate(**inputs, max_new_tokens=50)

print("\n[生成結果]:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

--- Installing libraries ---
--- Config 設定 ---
--- Loading Model and Tokenizer ---


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

{'loss': '5.21', 'grad_norm': '13.82', 'learning_rate': '0.0002', 'entropy': '2.22', 'mean_token_accuracy': '0.3539', 'num_tokens': '171', 'epoch': '1'}
{'loss': '4.134', 'grad_norm': '4.645', 'learning_rate': '0.00018', 'entropy': '2.594', 'mean_token_accuracy': '0.4133', 'num_tokens': '342', 'epoch': '2'}
{'loss': '3.458', 'grad_norm': '3.996', 'learning_rate': '0.00016', 'entropy': '2.66', 'mean_token_accuracy': '0.4733', 'num_tokens': '513', 'epoch': '3'}
{'loss': '2.981', 'grad_norm': '3.712', 'learning_rate': '0.00014', 'entropy': '2.579', 'mean_token_accuracy': '0.5152', 'num_tokens': '684', 'epoch': '4'}
{'loss': '2.643', 'grad_norm': '3.593', 'learning_rate': '0.00012', 'entropy': '2.42', 'mean_token_accuracy': '0.5356', 'num_tokens': '855', 'epoch': '5'}
{'loss': '2.321', 'grad_norm': '4.056', 'learning_rate': '0.0001', 'entropy': '2.309', 'mean_token_accuracy': '0.5897', 'num_tokens': '1026', 'epoch': '6'}
{'loss': '2.043', 'grad_norm': '4.27', 'learning_rate': '8e-05', 'ent